# Discovering Foreign Key Relationships in Databases Without Enforced Constraints

Many production databases lack explicitly defined foreign key constraints, whether due to performance considerations, legacy migrations, or flexible schema design. This analysis demonstrates a systematic approach to uncovering implicit relationships between tables using multiple detection methods:

1. **Column naming conventions** - Identifying patterns like `customer_id` that suggest references
2. **Data type compatibility** - Ensuring candidate columns have compatible types
3. **Value overlap analysis** - Measuring inclusion dependency between columns
4. **Cardinality profiling** - Determining relationship types (1:1, 1:M, M:M)

Each discovered relationship is assigned a confidence score based on how strongly these signals align.

## 1. Database Connection

The connection is configured automatically based on the database selected when running `main.py`. This allows the same notebook to work with different databases without manual configuration changes.

In [196]:
import pandas as pd
import numpy as np
import re
import os
import json
from collections import defaultdict

from setup_database import (
    connect_database, 
    get_table_names, 
    get_column_info,
    load_config
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

config = load_config()
DATABASE_TYPE = config['database_type']
DATABASE_NAME = config['database_name']

print(f"Database: {DATABASE_NAME}")
print(f"Type: {DATABASE_TYPE.upper()}")

Database: TPC-H Benchmark
Type: MYSQL


In [197]:
conn, IS_REMOTE, DATABASE_TYPE, config = connect_database(config)

def query(sql):
    """Execute SQL and return DataFrame."""
    return pd.read_sql(sql, conn)

Connected to MySQL: MYSQL1001.site4now.net/db_ac1b6f_tcbh
Note: Remote database - using optimized queries to minimize latency


## 2. Schema Discovery

Before identifying relationships, we catalog all tables with their columns, row counts, and data types. Data types are critical since a foreign key must be type-compatible with its referenced primary key.

In [198]:
table_names = get_table_names(conn, DATABASE_TYPE, config)
print(f"Found {len(table_names)} tables: {', '.join(table_names)}")


/home/shaden/Desktop/foreign-key-identification/setup_database.py:90: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SHOW TABLES", conn)


Found 8 tables: customer, lineitem, nation, orders, part, partsupp, region, supplier


In [199]:
table_info = []

print("Loading schema information...\n")

for table in table_names:
    # Use quoted identifiers for safety
    if DATABASE_TYPE == 'postgres':
        schema = config.get('postgres_schema', 'public')
        table_ref = f'"{schema}"."{table}"'
    else:
        table_ref = table
    
    row_count = query(f"SELECT COUNT(*) as cnt FROM {table_ref}")['cnt'][0]
    columns_with_types = get_column_info(conn, DATABASE_TYPE, config, table)
    
    table_info.append({
        'table_name': table,
        'row_count': row_count,
        'columns': [c[0] for c in columns_with_types],
        'column_types': {c[0]: c[1] for c in columns_with_types}
    })
    
    print(f"{table}: {row_count:,} rows, {len(columns_with_types)} columns")

total_rows = sum(t['row_count'] for t in table_info)
print(f"\nTotal: {total_rows:,} rows across {len(table_info)} tables")

Loading schema information...



/tmp/ipykernel_775390/3944372952.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)
/home/shaden/Desktop/foreign-key-identification/setup_database.py:108: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SHOW COLUMNS FROM {table_name}", conn)


customer: 30,000 rows, 8 columns
lineitem: 1,199,969 rows, 16 columns
nation: 25 rows, 4 columns
orders: 300,000 rows, 9 columns
part: 40,000 rows, 9 columns
partsupp: 160,000 rows, 5 columns
region: 5 rows, 3 columns
supplier: 2,000 rows, 7 columns

Total: 1,731,999 rows across 8 tables


In [200]:
for info in table_info:
    print(f"\n{info['table_name']}")
    for col in info['columns']:
        print(f"    {col}: {info['column_types'][col]}")


customer
    C_CUSTKEY: int
    C_NAME: varchar(25)
    C_ADDRESS: varchar(40)
    C_NATIONKEY: int
    C_PHONE: char(15)
    C_ACCTBAL: decimal(15,2)
    C_MKTSEGMENT: char(10)
    C_COMMENT: varchar(117)

lineitem
    L_ORDERKEY: int
    L_PARTKEY: int
    L_SUPPKEY: int
    L_LINENUMBER: int
    L_QUANTITY: decimal(15,2)
    L_EXTENDEDPRICE: decimal(15,2)
    L_DISCOUNT: decimal(15,2)
    L_TAX: decimal(15,2)
    L_RETURNFLAG: char(1)
    L_LINESTATUS: char(1)
    L_SHIPDATE: date
    L_COMMITDATE: date
    L_RECEIPTDATE: date
    L_SHIPINSTRUCT: char(25)
    L_SHIPMODE: char(10)
    L_COMMENT: varchar(44)

nation
    N_NATIONKEY: int
    N_NAME: char(25)
    N_REGIONKEY: int
    N_COMMENT: varchar(152)

orders
    O_ORDERKEY: int
    O_CUSTKEY: int
    O_ORDERSTATUS: char(1)
    O_TOTALPRICE: decimal(15,2)
    O_ORDERDATE: date
    O_ORDERPRIORITY: char(15)
    O_CLERK: char(15)
    O_SHIPPRIORITY: int
    O_COMMENT: varchar(79)

part
    P_PARTKEY: int
    P_NAME: varchar(55)
   

## 3. Foreign Key Column Detection

The first step in relationship discovery is identifying columns likely to be foreign keys. These typically follow naming conventions like `_id`, `_key`, or `_code` suffixes. While not every matching column is a foreign key, they form our candidate set for further validation.

In [201]:
FK_PATTERNS = ['_id', '_key', '_code', 'id', 'key']
EXCLUDED_COLUMNS = ['created_id', 'updated_id', 'uuid']

fk_candidates = []

for info in table_info:
    for col in info['columns']:
        col_lower = col.lower()
        
        if col_lower in [e.lower() for e in EXCLUDED_COLUMNS]:
            continue
        
        if any(pattern in col_lower for pattern in FK_PATTERNS):
            fk_candidates.append({
                'table': info['table_name'],
                'column': col,
                'data_type': info['column_types'][col]
            })

fk_df = pd.DataFrame(fk_candidates)
print(f"Identified {len(fk_candidates)} potential foreign key columns:\n")
print(fk_df.to_string(index=False))

Identified 15 potential foreign key columns:

   table      column data_type
customer   C_CUSTKEY       int
customer C_NATIONKEY       int
lineitem  L_ORDERKEY       int
lineitem   L_PARTKEY       int
lineitem   L_SUPPKEY       int
  nation N_NATIONKEY       int
  nation N_REGIONKEY       int
  orders  O_ORDERKEY       int
  orders   O_CUSTKEY       int
    part   P_PARTKEY       int
partsupp  PS_PARTKEY       int
partsupp  PS_SUPPKEY       int
  region R_REGIONKEY       int
supplier   S_SUPPKEY       int
supplier S_NATIONKEY       int


## 4. Relationship Candidate Generation

With FK candidates identified, we generate relationship hypotheses by matching each FK column against potential primary key columns in other tables. The matching uses multiple heuristics, each contributing to a confidence score:

- **Exact match**: Column names are identical after normalization
- **Base match**: Core names match after stripping prefixes/suffixes
- **Token overlap**: Shared meaningful tokens between names
- **Suffix alignment**: Final name components match
- **Table name correlation**: Column name relates to target table name

In [202]:
def normalize(name):
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')

def strip_prefix(col):
    parts = col.split('_')
    if len(parts) >= 2 and len(parts[0]) <= 2:
        return '_'.join(parts[1:])
    return col

def strip_suffix(col):
    col = normalize(col)
    for suffix in ['_id', '_key', 'id', 'key']:
        if col.endswith(suffix):
            return col[:-len(suffix)].rstrip('_')
    return col

def tokenize(s):
    return [t for t in normalize(s).split('_') if t]

def are_types_compatible(type1, type2):
    type1, type2 = type1.upper(), type2.upper()
    int_types = ['INT', 'INTEGER', 'BIGINT', 'SMALLINT', 'TINYINT']
    str_types = ['VARCHAR', 'CHAR', 'TEXT', 'STRING']
    
    def get_category(t):
        for int_t in int_types:
            if int_t in t:
                return 'integer'
        for str_t in str_types:
            if str_t in t:
                return 'string'
        return 'other'
    
    return get_category(type1) == get_category(type2)

In [203]:
def generate_relationship_candidates(table_info, fk_df):
    pk_candidates = {}
    for table in table_info:
        tname = table['table_name']
        pk_candidates[tname] = {
            col: table['column_types'][col]
            for col in table['columns']
            if normalize(col) == 'id' 
            or normalize(col).endswith('_id')
            or normalize(col).endswith('_key')
            or 'key' in normalize(col)
        }
    
    relationships = []
    
    for fk in fk_df.to_dict('records'):
        source_table = fk['table']
        source_col = fk['column']
        source_type = fk['data_type']
        
        source_norm = normalize(source_col)
        source_base = strip_suffix(strip_prefix(source_norm))
        source_tokens = tokenize(source_base)
        
        for target_table, pk_cols in pk_candidates.items():
            if target_table == source_table:
                continue
            
            for target_col, target_type in pk_cols.items():
                target_norm = normalize(target_col)
                target_base = strip_suffix(strip_prefix(target_norm))
                target_tokens = tokenize(target_base)
                
                score = 0
                rules = []
                
                if source_norm == target_norm:
                    score += 3
                    rules.append('exact_match')
                
                if source_base == target_base and source_base:
                    score += 3
                    rules.append('base_match')
                
                overlap = len(set(source_tokens) & set(target_tokens))
                if overlap > 0:
                    score += overlap
                    rules.append(f'token_overlap({overlap})')
                
                if source_norm.split('_')[-1] == target_norm.split('_')[-1]:
                    score += 2
                    rules.append('suffix_match')
                
                if strip_suffix(source_norm) in normalize(target_table):
                    score += 2
                    rules.append('table_name_match')
                
                type_compatible = are_types_compatible(source_type, target_type)
                
                if score > 0:
                    relationships.append({
                        'source_table': source_table,
                        'source_col': source_col,
                        'source_type': source_type,
                        'target_table': target_table,
                        'target_col': target_col,
                        'target_type': target_type,
                        'type_compatible': type_compatible,
                        'naming_score': score,
                        'naming_rules': rules
                    })
    
    if not relationships:
        return []
    
    rel_df = pd.DataFrame(relationships)
    best_matches = []
    
    for (src_t, src_c), group in rel_df.groupby(['source_table', 'source_col']):
        group_sorted = group.sort_values(
            ['type_compatible', 'naming_score'], 
            ascending=[False, False]
        )
        best_matches.append(group_sorted.iloc[0].to_dict())
    
    return best_matches

candidates = generate_relationship_candidates(table_info, fk_df)
print(f"Generated {len(candidates)} relationship candidates\n")

if candidates:
    for c in candidates:
        print(f"  {c['source_table']}.{c['source_col']} -> {c['target_table']}.{c['target_col']}")
        print(f"      score={c['naming_score']}, rules={c['naming_rules']}")

Generated 15 relationship candidates

  customer.C_CUSTKEY -> orders.O_CUSTKEY
      score=6, rules=['base_match', 'token_overlap(1)', 'suffix_match']
  customer.C_NATIONKEY -> nation.N_NATIONKEY
      score=6, rules=['base_match', 'token_overlap(1)', 'suffix_match']
  lineitem.L_ORDERKEY -> orders.O_ORDERKEY
      score=6, rules=['base_match', 'token_overlap(1)', 'suffix_match']
  lineitem.L_PARTKEY -> part.P_PARTKEY
      score=6, rules=['base_match', 'token_overlap(1)', 'suffix_match']
  lineitem.L_SUPPKEY -> partsupp.PS_SUPPKEY
      score=6, rules=['base_match', 'token_overlap(1)', 'suffix_match']
  nation.N_NATIONKEY -> customer.C_NATIONKEY
      score=6, rules=['base_match', 'token_overlap(1)', 'suffix_match']
  nation.N_REGIONKEY -> region.R_REGIONKEY
      score=6, rules=['base_match', 'token_overlap(1)', 'suffix_match']
  orders.O_CUSTKEY -> customer.C_CUSTKEY
      score=6, rules=['base_match', 'token_overlap(1)', 'suffix_match']
  orders.O_ORDERKEY -> lineitem.L_ORDERKEY
  

## 5. Statistical Validation

Naming conventions alone aren't sufficient. A column named `customer_id` might not actually reference the `customers` table if the data doesn't support it. We validate each candidate through:

**Inclusion Dependency**: What percentage of FK values exist in the referenced PK? A true foreign key should approach 100%.

**Cardinality Analysis**: Uniqueness of both columns determines relationship type (1:1, 1:M, M:M).

For remote databases, we optimize by fetching distinct values once per column, then performing comparisons locally to minimize network round trips.

In [204]:
SAMPLE_LIMIT = 50_000

def fetch_column_profile(table, column, row_count):
    profile_query = f"""
    SELECT 
        COUNT(DISTINCT {column}) as total_distinct,
        SUM(CASE WHEN {column} IS NULL THEN 1 ELSE 0 END) as null_count
    FROM {table}
    """
    
    values_query = f"""
    SELECT DISTINCT {column} as val
    FROM {table}
    WHERE {column} IS NOT NULL
    LIMIT {SAMPLE_LIMIT}
    """
    
    try:
        profile = query(profile_query).iloc[0]
        values_df = query(values_query)
        
        return {
            'total_distinct': int(profile['total_distinct']),
            'null_count': int(profile['null_count']),
            'values': set(values_df['val'].dropna().tolist()),
            'is_sampled': int(profile['total_distinct']) > SAMPLE_LIMIT
        }
    except Exception as e:
        print(f"    Error: {e}")
        return None

In [205]:
columns_to_profile = set()
for cand in candidates:
    columns_to_profile.add((cand['source_table'], cand['source_col']))
    columns_to_profile.add((cand['target_table'], cand['target_col']))

print(f"Fetching data for {len(columns_to_profile)} columns...")
if IS_REMOTE:
    print("(Remote database - this may take a few minutes)\n")
else:
    print()

column_profiles = {}
row_counts = {t['table_name']: t['row_count'] for t in table_info}

for i, (table, col) in enumerate(sorted(columns_to_profile)):
    print(f"  [{i+1}/{len(columns_to_profile)}] {table}.{col}", end="", flush=True)
    
    profile = fetch_column_profile(table, col, row_counts.get(table, 0))
    
    if profile:
        column_profiles[(table, col)] = profile
        sampled = " (sampled)" if profile['is_sampled'] else ""
        print(f" -> {profile['total_distinct']:,} distinct{sampled}")
    else:
        print(" -> FAILED")

print(f"\nProfiled {len(column_profiles)} columns successfully")

Fetching data for 15 columns...
(Remote database - this may take a few minutes)

  [1/15] customer.C_CUSTKEY

/tmp/ipykernel_775390/3944372952.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


 -> 30,000 distinct
  [2/15] customer.C_NATIONKEY -> 25 distinct
  [3/15] lineitem.L_ORDERKEY -> 300,000 distinct (sampled)
  [4/15] lineitem.L_PARTKEY -> 40,000 distinct
  [5/15] lineitem.L_SUPPKEY -> 2,000 distinct
  [6/15] nation.N_NATIONKEY -> 25 distinct
  [7/15] nation.N_REGIONKEY -> 5 distinct
  [8/15] orders.O_CUSTKEY -> 19,999 distinct
  [9/15] orders.O_ORDERKEY -> 300,000 distinct (sampled)
  [10/15] part.P_PARTKEY -> 40,000 distinct
  [11/15] partsupp.PS_PARTKEY -> 40,000 distinct
  [12/15] partsupp.PS_SUPPKEY -> 2,000 distinct
  [13/15] region.R_REGIONKEY -> 5 distinct
  [14/15] supplier.S_NATIONKEY -> 25 distinct
  [15/15] supplier.S_SUPPKEY -> 2,000 distinct

Profiled 15 columns successfully


In [206]:
print("Validating relationships...\n")

validated_relationships = []

for cand in candidates:
    src_key = (cand['source_table'], cand['source_col'])
    tgt_key = (cand['target_table'], cand['target_col'])
    
    src_profile = column_profiles.get(src_key)
    tgt_profile = column_profiles.get(tgt_key)
    
    if not src_profile or not tgt_profile:
        continue
    
    src_values = src_profile['values']
    tgt_values = tgt_profile['values']
    matched = src_values & tgt_values
    
    inclusion_pct = round(100.0 * len(matched) / len(src_values), 2) if src_values else 0
    orphaned = len(src_values) - len(matched)
    
    src_rows = row_counts.get(cand['source_table'], 0)
    tgt_rows = row_counts.get(cand['target_table'], 0)
    
    pk_unique = tgt_profile['total_distinct'] == tgt_rows
    fk_unique = src_profile['total_distinct'] == src_rows
    
    if pk_unique and fk_unique:
        cardinality = '1:1'
    elif pk_unique:
        cardinality = '1:M'
    elif fk_unique:
        cardinality = 'M:1'
    else:
        cardinality = 'M:M'
    
    confidence = inclusion_pct
    if not cand['type_compatible']:
        confidence *= 0.5
    if not pk_unique:
        confidence *= 0.7
    confidence = min(100, confidence + min(cand['naming_score'], 10))
    
    validated_relationships.append({
        'relationship': f"{cand['source_table']}.{cand['source_col']} -> {cand['target_table']}.{cand['target_col']}",
        'source_table': cand['source_table'],
        'source_col': cand['source_col'],
        'target_table': cand['target_table'],
        'target_col': cand['target_col'],
        'inclusion_pct': inclusion_pct,
        'cardinality': cardinality,
        'type_compatible': cand['type_compatible'],
        'naming_score': cand['naming_score'],
        'naming_rules': cand['naming_rules'],
        'confidence': round(confidence, 1),
        'orphaned_values': orphaned,
        'pk_is_unique': pk_unique,
        'source': 'naming'
    })

results_df = pd.DataFrame(validated_relationships)
results_df = results_df.sort_values('confidence', ascending=False)

print(f"Validated {len(results_df)} relationships")

Validating relationships...

Validated 15 relationships


## 6. Results Classification

Relationships are classified by confidence level:

- **High (90%+)**: Strong FK candidates with high value overlap and matching conventions
- **Medium (50-89%)**: Possible relationships requiring investigation; may indicate data quality issues
- **Low (<50%)**: Unlikely true relationships; probably false positives from naming patterns

In [207]:
high_conf = results_df[results_df['confidence'] >= 90]
med_conf = results_df[(results_df['confidence'] >= 50) & (results_df['confidence'] < 90)]
low_conf = results_df[results_df['confidence'] < 50]

print(f"High confidence (>=90%):   {len(high_conf)}")
print(f"Medium confidence (50-89%): {len(med_conf)}")
print(f"Low confidence (<50%):     {len(low_conf)}")

High confidence (>=90%):   5
Medium confidence (50-89%): 10
Low confidence (<50%):     0


In [208]:
if not high_conf.empty:
    print("High Confidence Relationships\n")
    cols = ['relationship', 'confidence', 'inclusion_pct', 'cardinality', 'naming_rules']
    print(high_conf[cols].to_string(index=False))
else:
    print("No high confidence relationships found.")

High Confidence Relationships

                              relationship  confidence  inclusion_pct cardinality                                 naming_rules
customer.C_NATIONKEY -> nation.N_NATIONKEY       100.0          100.0         1:M [base_match, token_overlap(1), suffix_match]
  lineitem.L_ORDERKEY -> orders.O_ORDERKEY       100.0          100.0         1:M [base_match, token_overlap(1), suffix_match]
      lineitem.L_PARTKEY -> part.P_PARTKEY       100.0          100.0         1:M [base_match, token_overlap(1), suffix_match]
  nation.N_REGIONKEY -> region.R_REGIONKEY       100.0          100.0         1:M [base_match, token_overlap(1), suffix_match]
    orders.O_CUSTKEY -> customer.C_CUSTKEY       100.0          100.0         1:M [base_match, token_overlap(1), suffix_match]


In [209]:
if not med_conf.empty:
    print("Medium Confidence Relationships\n")
    print("These may indicate data quality issues or partial relationships.\n")
    cols = ['relationship', 'confidence', 'inclusion_pct', 'orphaned_values', 'cardinality']
    print(med_conf[cols].to_string(index=False))

Medium Confidence Relationships

These may indicate data quality issues or partial relationships.

                                relationship  confidence  inclusion_pct  orphaned_values cardinality
  nation.N_NATIONKEY -> customer.C_NATIONKEY        76.0         100.00                0         M:1
   lineitem.L_SUPPKEY -> partsupp.PS_SUPPKEY        76.0         100.00                0         M:M
   partsupp.PS_SUPPKEY -> lineitem.L_SUPPKEY        76.0         100.00                0         M:M
    orders.O_ORDERKEY -> lineitem.L_ORDERKEY        76.0         100.00                0         M:1
        part.P_PARTKEY -> lineitem.L_PARTKEY        76.0         100.00                0         M:1
   partsupp.PS_PARTKEY -> lineitem.L_PARTKEY        76.0         100.00                0         M:M
supplier.S_NATIONKEY -> customer.C_NATIONKEY        76.0         100.00                0         M:M
    region.R_REGIONKEY -> nation.N_REGIONKEY        76.0         100.00                0     

## 7. Data Quality Analysis

For relationships with less than 100% inclusion, orphaned records indicate data quality issues. Understanding these helps determine whether a relationship is valid but has quality problems, or is a false positive.

In [210]:
relationships_with_orphans = results_df[results_df['orphaned_values'] > 0]

if not relationships_with_orphans.empty:
    print("Relationships with Orphaned Records\n")
    
    for _, row in relationships_with_orphans.head(10).iterrows():
        src_key = (row['source_table'], row['source_col'])
        tgt_key = (row['target_table'], row['target_col'])
        
        src_vals = column_profiles.get(src_key, {}).get('values', set())
        tgt_vals = column_profiles.get(tgt_key, {}).get('values', set())
        
        orphans = list(src_vals - tgt_vals)[:5]
        
        print(f"{row['relationship']}")
        print(f"    Orphaned: {row['orphaned_values']:,} values")
        print(f"    Sample: {orphans}")
        print()
else:
    print("No orphaned records found.")

Relationships with Orphaned Records

customer.C_CUSTKEY -> orders.O_CUSTKEY
    Orphaned: 10,001 values
    Sample: [3, 6, 9, 12, 15]



In [211]:
print("NULL Values in Foreign Key Columns\n")

null_data = []
for (table, col), profile in column_profiles.items():
    if profile['null_count'] > 0:
        row_count = row_counts.get(table, 1)
        null_pct = round(100.0 * profile['null_count'] / row_count, 2)
        null_data.append({
            'table': table,
            'column': col,
            'null_count': profile['null_count'],
            'null_pct': null_pct
        })

if null_data:
    null_df = pd.DataFrame(null_data).sort_values('null_pct', ascending=False)
    print(null_df.to_string(index=False))
else:
    print("No NULL values found in analyzed columns.")

NULL Values in Foreign Key Columns

No NULL values found in analyzed columns.


## 8. Final Results

Each detected relationship with its detection basis and confidence score.

In [212]:
def format_basis(row):
    parts = [
        f"column naming ({', '.join(row['naming_rules'])})",
        "compatible types" if row['type_compatible'] else "incompatible types",
        f"value overlap: {row['inclusion_pct']}%",
        f"cardinality: {row['cardinality']}"
    ]
    return '; '.join(parts)

print("DETECTED RELATIONSHIPS\n")

for _, row in results_df.iterrows():
    print(f"{row['relationship']}")
    print(f"    Confidence: {row['confidence']}%")
    print(f"    Basis: {format_basis(row)}")
    print()

DETECTED RELATIONSHIPS

customer.C_NATIONKEY -> nation.N_NATIONKEY
    Confidence: 100.0%
    Basis: column naming (base_match, token_overlap(1), suffix_match); compatible types; value overlap: 100.0%; cardinality: 1:M

lineitem.L_ORDERKEY -> orders.O_ORDERKEY
    Confidence: 100.0%
    Basis: column naming (base_match, token_overlap(1), suffix_match); compatible types; value overlap: 100.0%; cardinality: 1:M

lineitem.L_PARTKEY -> part.P_PARTKEY
    Confidence: 100.0%
    Basis: column naming (base_match, token_overlap(1), suffix_match); compatible types; value overlap: 100.0%; cardinality: 1:M

nation.N_REGIONKEY -> region.R_REGIONKEY
    Confidence: 100.0%
    Basis: column naming (base_match, token_overlap(1), suffix_match); compatible types; value overlap: 100.0%; cardinality: 1:M

orders.O_CUSTKEY -> customer.C_CUSTKEY
    Confidence: 100.0%
    Basis: column naming (base_match, token_overlap(1), suffix_match); compatible types; value overlap: 100.0%; cardinality: 1:M

nation.N_N

In [213]:
export_df = results_df.copy()
export_df['detection_basis'] = results_df.apply(format_basis, axis=1)

export_file = f"discovered_relationships_{DATABASE_TYPE}.csv"
export_df.to_csv(export_file, index=False)
print(f"Results exported to {export_file}")

Results exported to discovered_relationships_mysql.csv


In [214]:
print("ANALYSIS SUMMARY\n")
print(f"Database: {DATABASE_NAME} ({DATABASE_TYPE.upper()})")
print(f"Tables: {len(table_names)}")
print(f"Total rows: {total_rows:,}")
print()
print(f"FK candidates identified: {len(fk_candidates)}")
print(f"Relationships tested: {len(candidates)}")
print(f"Relationships validated: {len(results_df)}")
print()
print(f"High confidence: {len(high_conf)}")
print(f"Medium confidence: {len(med_conf)}")
print(f"Low confidence: {len(low_conf)}")

ANALYSIS SUMMARY

Database: TPC-H Benchmark (MYSQL)
Tables: 8
Total rows: 1,731,999

FK candidates identified: 15
Relationships tested: 15
Relationships validated: 15

High confidence: 5
Medium confidence: 10
Low confidence: 0


## 9. Data Profiling with ydata-profiling

Before diving into relationship discovery, it's often valuable to understand the shape and quality of your data comprehensively. While our analysis above targets specific columns, a full profiling step can reveal patterns we might otherwise miss.

I chose `ydata-profiling` (formerly `pandas-profiling`) for this task because it generates interactive HTML reports with minimal code. For FK discovery specifically, profiling helps identify:

- **High cardinality columns** that might be primary keys we overlooked
- **Unexpected data types** that could cause join failures
- **Missing value patterns** that explain low inclusion percentages
- **Value distributions** that suggest whether a column is truly an identifier

In [215]:
try:
    from ydata_profiling import ProfileReport
    PROFILING_AVAILABLE = True
    print("ydata-profiling is available")
except ImportError:
    print("ydata-profiling not installed.")
    print("To enable profiling, run: pip install ydata-profiling")
    PROFILING_AVAILABLE = False

ydata-profiling is available


In [216]:
def generate_table_profile(table_name, sample_size=100000):
    if not PROFILING_AVAILABLE:
        return None
    
    row_count = row_counts.get(table_name, 0)
    
    if row_count > sample_size:
        if DATABASE_TYPE == 'sqlite':
            df = pd.read_sql(f"SELECT * FROM {table_name} ORDER BY RANDOM() LIMIT {sample_size}", conn)
        else:
            df = pd.read_sql(f"SELECT * FROM {table_name} ORDER BY RAND() LIMIT {sample_size}", conn)
        title = f"{table_name} (sampled: {sample_size:,} of {row_count:,} rows)"
    else:
        df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
        title = f"{table_name} ({row_count:,} rows)"
    
    return ProfileReport(df, title=title, minimal=True, correlations=None, explorative=True)


def profile_key_tables(tables_to_profile=None, output_dir='profiles'):
    if not PROFILING_AVAILABLE:
        print("Profiling not available. Install ydata-profiling to enable.")
        return
    
    os.makedirs(output_dir, exist_ok=True)
    
    if tables_to_profile is None:
        tables_to_profile = set()
        if not high_conf.empty:
            for _, row in high_conf.iterrows():
                tables_to_profile.add(row['source_table'])
                tables_to_profile.add(row['target_table'])
        tables_to_profile = list(tables_to_profile)
    
    print(f"Generating profiles for {len(tables_to_profile)} tables...\n")
    
    for table in tables_to_profile:
        print(f"  Profiling {table}...", end=" ", flush=True)
        try:
            profile = generate_table_profile(table)
            if profile:
                profile.to_file(f"{output_dir}/{table}_profile.html")
                print(f"saved")
        except Exception as e:
            print(f"failed: {e}")
    
    print(f"\nProfiles saved to '{output_dir}/' directory")

# Uncomment to run:
# profile_key_tables()

## 10. Exhaustive Column Analysis

The approach we've taken starts with naming conventions to identify FK candidates, then validates them through statistical analysis. This is efficient but has a blind spot: it misses relationships where columns don't follow naming conventions. In legacy systems, foreign keys might be named `cust` instead of `customer_id`.

An alternative is to test **all columns** against each other, relying purely on data characteristics rather than naming. This is computationally expensive (O(N²) column pairs), which is why I didn't use it as the primary approach.

However, running this analysis provides two benefits:
1. **Catches hidden relationships** that naming conventions missed
2. **Provides more diverse samples for ML training** - relationships with `naming_score=0` but high inclusion create feature variety that helps ML models learn what actually matters

In [217]:
def find_hidden_relationships(table_info, existing_fk_cols, column_profiles, row_counts, min_inclusion=50):
    """
    Test columns not identified by naming conventions.
    Uses lower inclusion threshold to capture more samples for ML training.
    """
    tested_cols = set((c['table'], c['column']) for c in existing_fk_cols)
    
    potential_keys = []
    for info in table_info:
        for col in info['columns']:
            if (info['table_name'], col) in tested_cols:
                continue
            col_type = info['column_types'][col].upper()
            if any(t in col_type for t in ['TEXT', 'BLOB', 'DATE', 'TIME', 'FLOAT', 'DOUBLE', 'DECIMAL']):
                continue
            potential_keys.append({
                'table': info['table_name'],
                'column': col,
                'data_type': info['column_types'][col],
                'row_count': info['row_count']
            })
    
    print(f"Found {len(potential_keys)} columns not tested by naming conventions")
    
    if not potential_keys:
        return [], {}
    
    if len(potential_keys) > 35:
        print(f"Limiting to 35 columns for performance")
        potential_keys = potential_keys[:35]
    
    print("Fetching column profiles...\n")
    
    new_profiles = {}
    for i, pk in enumerate(potential_keys):
        key = (pk['table'], pk['column'])
        if key in column_profiles:
            new_profiles[key] = column_profiles[key]
        else:
            print(f"  [{i+1}/{len(potential_keys)}] {pk['table']}.{pk['column']}...", end="", flush=True)
            profile = fetch_column_profile(pk['table'], pk['column'], pk['row_count'])
            if profile:
                new_profiles[key] = profile
                print(f" {profile['total_distinct']:,} distinct")
            else:
                print(" skipped")
    
    all_profiles = {**column_profiles, **new_profiles}
    
    print(f"\nTesting value overlap...")
    
    pk_columns = [(info['table_name'], col, info['column_types'][col])
                  for info in table_info for col in info['columns']
                  if normalize(col) == 'id' or normalize(col).endswith('_id') or normalize(col).endswith('_key')]
    
    hidden = []
    for pk in potential_keys:
        src_key = (pk['table'], pk['column'])
        src_profile = all_profiles.get(src_key)
        if not src_profile or not src_profile['values']:
            continue
        
        for tgt_table, tgt_col, tgt_type in pk_columns:
            if tgt_table == pk['table']:
                continue
            if not are_types_compatible(pk['data_type'], tgt_type):
                continue
            
            tgt_profile = all_profiles.get((tgt_table, tgt_col))
            if not tgt_profile or not tgt_profile['values']:
                continue
            
            matched = src_profile['values'] & tgt_profile['values']
            inclusion_pct = 100.0 * len(matched) / len(src_profile['values'])
            
            if inclusion_pct >= min_inclusion:
                tgt_rows = row_counts.get(tgt_table, 1)
                src_rows = row_counts.get(pk['table'], 1)
                pk_unique = tgt_profile['total_distinct'] == tgt_rows
                fk_unique = src_profile['total_distinct'] == src_rows
                
                if pk_unique and fk_unique: cardinality = '1:1'
                elif pk_unique: cardinality = '1:M'
                elif fk_unique: cardinality = 'M:1'
                else: cardinality = 'M:M'
                
                confidence = inclusion_pct
                if not are_types_compatible(pk['data_type'], tgt_type):
                    confidence *= 0.5
                if not pk_unique:
                    confidence *= 0.7
                
                hidden.append({
                    'relationship': f"{pk['table']}.{pk['column']} -> {tgt_table}.{tgt_col}",
                    'source_table': pk['table'],
                    'source_col': pk['column'],
                    'target_table': tgt_table,
                    'target_col': tgt_col,
                    'inclusion_pct': round(inclusion_pct, 2),
                    'cardinality': cardinality,
                    'type_compatible': are_types_compatible(pk['data_type'], tgt_type),
                    'naming_score': 0,
                    'naming_rules': [],
                    'confidence': round(confidence, 1),
                    'orphaned_values': len(src_profile['values']) - len(matched),
                    'pk_is_unique': pk_unique,
                    'source': 'exhaustive'
                })
    
    return hidden, all_profiles


print("Searching for hidden FK relationships...")
print("(Testing columns that don't follow naming conventions)\n")

hidden_fks, extended_profiles = find_hidden_relationships(
    table_info, fk_candidates, column_profiles, row_counts, min_inclusion=50
)

column_profiles.update(extended_profiles)

print(f"\nFound {len(hidden_fks)} potential relationships from exhaustive analysis")

if hidden_fks:
    hidden_df = pd.DataFrame(hidden_fks).sort_values('confidence', ascending=False)
    high_conf_hidden = hidden_df[hidden_df['confidence'] >= 80]
    print(f"  High confidence (>=80%): {len(high_conf_hidden)}")
    print(f"  Lower confidence (<80%): {len(hidden_fks) - len(high_conf_hidden)}")
    
    if not high_conf_hidden.empty:
        print("\nHigh confidence hidden relationships:\n")
        print(high_conf_hidden[['relationship', 'confidence', 'inclusion_pct', 'cardinality']].to_string(index=False))
else:
    hidden_df = pd.DataFrame()

Searching for hidden FK relationships...
(Testing columns that don't follow naming conventions)

Found 33 columns not tested by naming conventions
Fetching column profiles...

  [1/33] customer.C_NAME...

/tmp/ipykernel_775390/3944372952.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


 30,000 distinct
  [2/33] customer.C_ADDRESS... 30,000 distinct
  [3/33] customer.C_PHONE... 30,000 distinct
  [4/33] customer.C_MKTSEGMENT... 5 distinct
  [5/33] customer.C_COMMENT... 29,999 distinct
  [6/33] lineitem.L_LINENUMBER... 7 distinct
  [7/33] lineitem.L_RETURNFLAG... 3 distinct
  [8/33] lineitem.L_LINESTATUS... 2 distinct
  [9/33] lineitem.L_SHIPINSTRUCT... 4 distinct
  [10/33] lineitem.L_SHIPMODE... 7 distinct
  [11/33] lineitem.L_COMMENT... 1,019,497 distinct
  [12/33] nation.N_NAME... 25 distinct
  [13/33] nation.N_COMMENT... 25 distinct
  [14/33] orders.O_ORDERSTATUS... 3 distinct
  [15/33] orders.O_ORDERPRIORITY... 5 distinct
  [16/33] orders.O_CLERK... 1,000 distinct
  [17/33] orders.O_SHIPPRIORITY... 1 distinct
  [18/33] orders.O_COMMENT... 298,857 distinct
  [19/33] part.P_NAME... 40,000 distinct
  [20/33] part.P_MFGR... 5 distinct
  [21/33] part.P_BRAND... 25 distinct
  [22/33] part.P_TYPE... 150 distinct
  [23/33] part.P_SIZE... 50 distinct
  [24/33] part.P_CONTAI

##### In this database, the naming conventions are consistent, so no hidden relationships were found. However, the pipeline is designed to handle messier schemas where this step would surface additional FK candidates and feed them into the ML training set for better feature diversity.

In [218]:
# Combine naming-based and exhaustive results for ML training
if not hidden_df.empty:
    combined_df = pd.concat([results_df, hidden_df], ignore_index=True)
    combined_df = combined_df.drop_duplicates(subset=['relationship'], keep='first')
    combined_df = combined_df.sort_values('confidence', ascending=False)
    
    print(f"Combined dataset for ML training:")
    print(f"  From naming conventions: {len(results_df)}")
    print(f"  From exhaustive analysis: {len(hidden_df)}")
    print(f"  Total unique: {len(combined_df)}")
else:
    combined_df = results_df.copy()
    print(f"Using {len(combined_df)} relationships from naming-based analysis.")

Using 15 relationships from naming-based analysis.


## 11. Exploring Machine Learning Approaches

The rule-based approach works well for databases with standard naming patterns, as we've seen in this analysis. However, the confidence scoring relies on manually tuned weights, and the approach may struggle with inconsistent schemas.

This section explores how machine learning could improve FK discovery. The pipeline is designed to combine results from both naming-based and exhaustive column analysis when available, creating a diverse training set with:

- **High naming scores + high inclusion**: Clear FK relationships from naming analysis
- **Zero naming scores + high inclusion**: Hidden FKs that would come from exhaustive analysis
- **Various confidence levels**: Both strong matches and weaker candidates

In this database, all relationships followed naming conventions, so our training data comes entirely from the naming-based analysis. For databases with less consistent naming, the exhaustive analysis would contribute additional samples with `naming_score=0`, helping the model learn to distinguish true FKs based on data characteristics alone rather than relying solely on column names.

Despite having only naming-based samples here, the ML exploration still demonstrates the methodology and reveals which features are most predictive for this schema.

In [219]:
try:
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    from sklearn.model_selection import cross_val_score
    ML_AVAILABLE = True
    print("scikit-learn is available")
except ImportError:
    print("scikit-learn not installed. Run: pip install scikit-learn")
    ML_AVAILABLE = False

scikit-learn is available


In [220]:
def extract_ml_features(df, column_profiles, row_counts):
    features = []
    for _, row in df.iterrows():
        src_profile = column_profiles.get((row['source_table'], row['source_col']), {})
        tgt_profile = column_profiles.get((row['target_table'], row['target_col']), {})
        src_rows = row_counts.get(row['source_table'], 1)
        tgt_rows = row_counts.get(row['target_table'], 1)
        
        rules = row['naming_rules'] if isinstance(row['naming_rules'], list) else []
        src_vals = src_profile.get('values', set())
        
        features.append({
            'relationship': row['relationship'],
            'naming_score': row['naming_score'],
            'has_exact_match': 1 if 'exact_match' in rules else 0,
            'has_base_match': 1 if 'base_match' in rules else 0,
            'has_table_match': 1 if 'table_name_match' in rules else 0,
            'type_compatible': 1 if row['type_compatible'] else 0,
            'inclusion_pct': row['inclusion_pct'],
            'orphan_ratio': row['orphaned_values'] / max(len(src_vals), 1),
            'src_uniqueness': src_profile.get('total_distinct', 0) / max(src_rows, 1),
            'tgt_uniqueness': tgt_profile.get('total_distinct', 0) / max(tgt_rows, 1),
            'cardinality_ratio': src_profile.get('total_distinct', 0) / max(tgt_profile.get('total_distinct', 1), 1),
            'pk_is_unique': 1 if row['pk_is_unique'] else 0,
            'src_null_ratio': src_profile.get('null_count', 0) / max(src_rows, 1),
            'tgt_null_ratio': tgt_profile.get('null_count', 0) / max(tgt_rows, 1),
            'confidence': row['confidence'],
            'source': row.get('source', 'naming')
        })
    return pd.DataFrame(features)

if ML_AVAILABLE and not combined_df.empty:
    ml_features_df = extract_ml_features(combined_df, column_profiles, row_counts)
    print(f"Extracted features for {len(ml_features_df)} candidates")
    print(f"  From naming: {(ml_features_df['source'] == 'naming').sum()}")
    print(f"  From exhaustive: {(ml_features_df['source'] == 'exhaustive').sum()}")
else:
    ml_features_df = pd.DataFrame()

Extracted features for 15 candidates
  From naming: 15
  From exhaustive: 0


In [221]:
def find_balanced_threshold(values):
    sorted_v = sorted(values)
    n = len(sorted_v)
    min_per_class = max(2, n // 4)
    for i in range(min_per_class, n - min_per_class):
        return sorted_v[i]
    return None

def train_models(features_df):
    if not ML_AVAILABLE:
        return None, None, None
    
    feature_cols = ['naming_score', 'has_exact_match', 'has_base_match', 'has_table_match',
                    'type_compatible', 'inclusion_pct', 'orphan_ratio', 'src_uniqueness',
                    'tgt_uniqueness', 'cardinality_ratio', 'pk_is_unique', 'src_null_ratio', 'tgt_null_ratio']
    
    df = features_df.copy()
    X = df[feature_cols].fillna(0)
    
    threshold = find_balanced_threshold(df['confidence'].values)
    
    if threshold:
        df['is_fk'] = (df['confidence'] >= threshold).astype(int)
        y = df['is_fk']
        pos, neg = y.sum(), len(y) - y.sum()
        
        if pos >= 2 and neg >= 2:
            print(f"Classification with threshold {threshold:.1f}%")
            print(f"  Positive: {pos}, Negative: {neg}\n")
            
            models = {
                'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
                'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
            }
            results = {}
            for name, model in models.items():
                cv = cross_val_score(model, X, y, cv=min(3, pos, neg), scoring='accuracy')
                model.fit(X, y)
                results[name] = {'model': model, 'cv_mean': cv.mean(), 'cv_std': cv.std(),
                                 'feature_importance': dict(zip(feature_cols, model.feature_importances_))}
            return results, feature_cols, 'classification'
    
    print("Using regression (class imbalance)\n")
    y = df['confidence']
    models = {
        'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
    }
    results = {}
    for name, model in models.items():
        cv = cross_val_score(model, X, y, cv=min(3, len(y)), scoring='r2')
        model.fit(X, y)
        results[name] = {'model': model, 'cv_mean': cv.mean(), 'cv_std': cv.std(),
                         'feature_importance': dict(zip(feature_cols, model.feature_importances_))}
    return results, feature_cols, 'regression'

if ML_AVAILABLE and len(ml_features_df) >= 4:
    ml_results, feature_cols, model_type = train_models(ml_features_df)
    if ml_results:
        print("Model Performance:")
        for name, r in ml_results.items():
            metric = 'Accuracy' if model_type == 'classification' else 'R²'
            print(f"  {name}: {metric} = {r['cv_mean']:.2f} (+/- {r['cv_std']:.2f})")
else:
    ml_results, feature_cols, model_type = None, [], None

Using regression (class imbalance)

Model Performance:
  Random Forest: R² = -0.08 (+/- 0.12)
  Gradient Boosting: R² = -0.08 (+/- 0.12)


In [222]:
if ml_results:
    print("Feature Importance (Random Forest)\n")
    importance = ml_results['Random Forest']['feature_importance']
    for feat, imp in sorted(importance.items(), key=lambda x: -x[1]):
        print(f"  {feat:20s} {imp:.3f} {'*' * int(imp * 40)}")

Feature Importance (Random Forest)

  tgt_uniqueness       0.485 *******************
  pk_is_unique         0.381 ***************
  inclusion_pct        0.058 **
  cardinality_ratio    0.052 **
  orphan_ratio         0.024 
  naming_score         0.000 
  has_exact_match      0.000 
  has_base_match       0.000 
  has_table_match      0.000 
  type_compatible      0.000 
  src_uniqueness       0.000 
  src_null_ratio       0.000 
  tgt_null_ratio       0.000 


In [223]:
if ml_results and feature_cols:
    rf = ml_results['Random Forest']['model']
    X = ml_features_df[feature_cols].fillna(0)
    
    if model_type == 'classification':
        ml_features_df['ml_score'] = (rf.predict_proba(X)[:, 1] * 100).round(1)
        label = 'ML Probability'
    else:
        ml_features_df['ml_score'] = rf.predict(X).round(1)
        label = 'ML Predicted'
    
    comp = ml_features_df[['relationship', 'confidence', 'ml_score', 'source']].copy()
    comp.columns = ['relationship', 'Rule-Based', label, 'source']
    print(f"Rule-Based vs {label}\n")
    print(comp.sort_values(label, ascending=False).to_string(index=False))

Rule-Based vs ML Predicted

                                relationship  Rule-Based  ML Predicted source
  customer.C_NATIONKEY -> nation.N_NATIONKEY       100.0         100.0 naming
    lineitem.L_ORDERKEY -> orders.O_ORDERKEY       100.0         100.0 naming
        lineitem.L_PARTKEY -> part.P_PARTKEY       100.0         100.0 naming
    nation.N_REGIONKEY -> region.R_REGIONKEY       100.0         100.0 naming
      orders.O_CUSTKEY -> customer.C_CUSTKEY       100.0         100.0 naming
  nation.N_NATIONKEY -> customer.C_NATIONKEY        76.0          76.0 naming
   lineitem.L_SUPPKEY -> partsupp.PS_SUPPKEY        76.0          76.0 naming
   partsupp.PS_SUPPKEY -> lineitem.L_SUPPKEY        76.0          76.0 naming
        part.P_PARTKEY -> lineitem.L_PARTKEY        76.0          76.0 naming
   partsupp.PS_PARTKEY -> lineitem.L_PARTKEY        76.0          76.0 naming
supplier.S_NATIONKEY -> customer.C_NATIONKEY        76.0          76.0 naming
    supplier.S_SUPPKEY -> lineitem.L

##### ML EXPLORATION SUMMARY

Demonstrated:
  - Combined naming + exhaustive analysis for diverse training data
  - Adaptive thresholding / regression fallback
  - Feature importance analysis

For production: collect ground truth labels from known FK schemas.

In [224]:
conn.close()
print("Database connection closed.")

Database connection closed.
